# 🎙️ AI Haberleri - Otomatik Podcast Üretici

Bu notebook, günün AI haberlerini yüksek kaliteli Türkçe sese dönüştürür.

**Model:** XTTS v2 (Coqui TTS) - Türkçe Neural Voice

**Özellikler:**
- Voice Cloning desteği
- Doğal Türkçe telaffuz
- Ücretsiz GPU kullanımı

In [ ]:
# 🔧 GPU Kontrol
!nvidia-smi

import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# 📦 Bağımlılıkları Kur (İlk çalıştırmada ~2-3 dk)
!pip install -q TTS==0.22.0
!pip install -q gdown requests python-dotenv

In [ ]:
# 🎤 XTTS v2 Modelini Yükle
from TTS.api import TTS
import torch

# XTTS v2 - En iyi Türkçe destekli model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Model yükleniyor (~1-2 dk)
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("✅ XTTS v2 model yüklendi!")

In [ ]:
# 🎙️ Örnek Türkçe Ses Dosyası İndir (Voice Cloning için)
# Kendi sesinizi kullanmak için bu hücreyi değiştirin

import os
import urllib.request

# Örnek Türkçe erkek sesi (Wikipedia'dan)
VOICE_SAMPLE_URL = "https://upload.wikimedia.org/wikipedia/commons/8/8e/Tr-Merhaba.ogg"
VOICE_SAMPLE_PATH = "/content/reference_voice.wav"

# Kendi ses dosyanızı yüklemek için:
# from google.colab import files
# uploaded = files.upload()
# VOICE_SAMPLE_PATH = list(uploaded.keys())[0]

# Örnek sesi indir
if not os.path.exists(VOICE_SAMPLE_PATH):
    !wget -q -O /content/reference_voice.ogg "$VOICE_SAMPLE_URL"
    !ffmpeg -y -i /content/reference_voice.ogg -ar 22050 -ac 1 "$VOICE_SAMPLE_PATH" 2>/dev/null
    print("✅ Referans ses dosyası hazır!")
else:
    print("✅ Referans ses zaten mevcut.")

In [ ]:
# 📰 AI Haberleri API'den Çek
import requests
from datetime import datetime

# API URL - Kendi sitenizin URL'sini girin
API_URL = "https://aihaberleri.org/api/articles"
# Alternatif: Doğrudan metin girin
MANUAL_TEXT = None  # Boş bırakın API kullanmak için


def get_todays_news(limit=5):
    """Günün en son haberlerini çek"""
    try:
        response = requests.get(f"{API_URL}?limit={limit}&status=PUBLISHED", timeout=30)
        if response.status_code == 200:
            data = response.json()
            articles = data.get("articles", data) if isinstance(data, dict) else data
            return articles[:limit]
    except Exception as e:
        print(f"API Hatası: {e}")
    return []


def create_podcast_script(articles):
    """Haberlerden podcast scripti oluştur"""
    today = datetime.now().strftime("%d %B %Y")

    script = f"""Merhaba, AI Haberleri podcast'ine hoş geldiniz. 
Ben yapay zeka asistanınız. Bugün {today}, sizler için günün en önemli yapay zeka haberlerini derledim.

"""

    for i, article in enumerate(articles, 1):
        title = article.get("title", "")
        summary = article.get("summary", article.get("content", ""))[:500]
        script += f"""Haber {i}: {title}.
{summary}

"""

    script += """Bu günkü haberlerimiz bu kadardı. 
Bizi dinlediğiniz için teşekkür ederiz. Yarın yeni haberlerle tekrar görüşmek üzere, hoşça kalın!"""

    return script


# Haberleri çek veya manuel metin kullan
if MANUAL_TEXT:
    podcast_script = MANUAL_TEXT
    print("📝 Manuel metin kullanılıyor.")
else:
    news = get_todays_news(5)
    if news:
        podcast_script = create_podcast_script(news)
        print(f"📰 {len(news)} haber çekildi.")
    else:
        # Örnek metin
        podcast_script = """Merhaba, AI Haberleri podcast'ine hoş geldiniz.
Bugün yapay zeka dünyasından önemli gelişmeleri sizlerle paylaşacağız.

İlk haberimiz: OpenAI, yeni GPT-5 modelini duyurdu. 
Bu model, önceki versiyonlara göre çok daha gelişmiş muhakeme yeteneklerine sahip.

İkinci haberimiz: Google DeepMind, AlphaFold 3'ü açık kaynak olarak yayınladı.
Bilim insanları artık protein yapılarını ücretsiz olarak tahmin edebilecek.

Bu günkü haberlerimiz bu kadardı. Yarın yeni haberlerle görüşmek üzere!"""
        print("⚠️ API'ye ulaşılamadı, örnek metin kullanılıyor.")

print("=" * 50)
print("📃 PODCAST SCRİPTİ:")
print("=" * 50)
print(podcast_script[:1000] + "..." if len(podcast_script) > 1000 else podcast_script)

In [ ]:
# 🔊 Podcast Sesini Üret (XTTS v2)
import time
from datetime import datetime

OUTPUT_DIR = "/content/podcasts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Dosya adı
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"{OUTPUT_DIR}/podcast_{timestamp}.wav"

print("🎙️ Ses üretiliyor... (Bu 2-5 dakika sürebilir)")
start_time = time.time()

# XTTS ile ses üret
try:
    tts.tts_to_file(
        text=podcast_script,
        file_path=output_file,
        speaker_wav=VOICE_SAMPLE_PATH,
        language="tr",
        split_sentences=True,
    )

    elapsed = time.time() - start_time
    file_size = os.path.getsize(output_file) / (1024 * 1024)

    print(f"\n✅ Podcast oluşturuldu!")
    print(f"📁 Dosya: {output_file}")
    print(f"📏 Boyut: {file_size:.2f} MB")
    print(f"⏱️ Süre: {elapsed:.1f} saniye")

except Exception as e:
    print(f"❌ Hata: {e}")

In [ ]:
# 🎧 Sesi Dinle
from IPython.display import Audio, display

if os.path.exists(output_file):
    display(Audio(output_file))
else:
    print("❌ Ses dosyası bulunamadı.")

In [ ]:
# 💾 MP3'e Dönüştür ve İndir
mp3_file = output_file.replace(".wav", ".mp3")

# WAV -> MP3 dönüştürme
!ffmpeg -y -i "{output_file}" -codec:a libmp3lame -qscale:a 2 "{mp3_file}" 2>/dev/null

if os.path.exists(mp3_file):
    mp3_size = os.path.getsize(mp3_file) / (1024 * 1024)
    print(f"✅ MP3 oluşturuldu: {mp3_file}")
    print(f"📏 Boyut: {mp3_size:.2f} MB")

    # Bilgisayara indir
    from google.colab import files

    files.download(mp3_file)
else:
    print("❌ MP3 dönüştürme başarısız.")

In [ ]:
# ☁️ Google Drive'a Kaydet (Opsiyonel)
SAVE_TO_DRIVE = True  # True yaparak Drive'a kaydedin

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

    # Podcast klasörü oluştur
    drive_folder = "/content/drive/MyDrive/AI_Haberleri_Podcasts"
    os.makedirs(drive_folder, exist_ok=True)

    # MP3'i kopyala
    import shutil

    drive_mp3 = f"{drive_folder}/podcast_{timestamp}.mp3"
    shutil.copy(mp3_file, drive_mp3)

    print(f"✅ Drive'a kaydedildi: {drive_mp3}")

---
## 🤖 Otomatik Çalıştırma (Opsiyonel)

Bu notebook'u otomatik çalıştırmak için:

### Yöntem 1: Colab Pro + Scheduled Execution
1. Colab Pro satın alın ($10/ay)
2. Runtime → Schedule notebook execution

### Yöntem 2: GitHub Actions + Colab API
1. Bu notebook'u `.ipynb` olarak GitHub'a yükleyin
2. GitHub Actions ile günlük tetikleyin
3. `papermill` kullanarak parametreli çalıştırın

### Yöntem 3: Modal.com (Önerilen - Ücretsiz GPU)
Modal.com'da bu kodu serverless olarak çalıştırın.
Ayda 30$ kredi ücretsiz!